# Shamela HF Ingestion & Filtering

This notebook demonstrates how to stream, filter, and save books from the Hugging Face `ieasybooks-org/shamela-waqfeya-library` dataset using Prefect tasks.

In [ ]:
from datasets import load_dataset
import pandas as pd
import os
import sys

# Add the parent directory to path to import local tasks if needed
sys.path.append('..')

## 1. Inspect the Dataset Schema
Before filtering, let's see what the columns are.

In [ ]:
dataset_name = "ieasybooks-org/shamela-waqfeya-library"
dataset = load_dataset(dataset_name, split="train", streaming=True)
first_row = next(iter(dataset))
print(f"Columns: {list(first_row.keys())}")

## 2. Define Prefect Tasks
We wrap the logic in Prefect tasks for orchestration.

In [ ]:
from prefect import task, flow

@task
def filter_books(category):
    dataset = load_dataset(dataset_name, split="train", streaming=True)
    filtered = []
    for row in dataset:
        if row.get('category') == category:
            filtered.append(row)
            if len(filtered) >= 10: # Limit for demo
                break
    return filtered

@task
def save_books(books, path):
    df = pd.DataFrame(books)
    df.to_parquet(path)
    return path

## 3. Run the Flow
You can run this locally or register it with the Prefect server.

In [ ]:
@flow
def ingestion_flow(target_category="التفاسير"):
    books = filter_books(target_category)
    path = f"shamela_{target_category}.parquet"
    save_books(books, path)
    print(f"Success! Saved {len(books)} books to {path}")

ingestion_flow()

## 4. Query with DuckDB (The Pro Way)
Once saved, querying is instantaneous.

In [ ]:
import duckdb
target_category = "التفاسير"
path = f"shamela_{target_category}.parquet"
if os.path.exists(path):
    res = duckdb.query(f"SELECT title, category FROM '{path}' LIMIT 5").df()
    display(res)
else:
    print("File not found. Please run the flow first.")